# Mini EDA Project — Employee Dataset

**Objective:** explore `data/sample_dataset_W3D5.csv`, clean it, and pull
out key findings about department composition, salary, experience, and
satisfaction — supported by charts.

This notebook assumes it's run from the project's `notebooks/` folder,
with `../data/` and `../plots/` as siblings.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid")
os.makedirs("../plots", exist_ok=True)

df = pd.read_csv("../data/sample_dataset_W3D5.csv")
print("Shape:", df.shape)
df.head()

Shape: (256, 6)


,employee_name,department,years_experience,salary,satisfaction_score,hire_date
0,Employee_054,Sales,5.0,"73,592",6.7,2021-12-26
1,Employee_206,Engineering,5.4,"92,710",7.4,2019-09-06
2,Employee_036,Engineering,8.7,"103,736",5.9,2019-02-11
3,Employee_011,ENGINEERING,10.7,"104,421",7.4,2023-08-08
4,Employee_189,Marketing,8.3,"75,135",8.7,2016-03-21


## 1. Initial Inspection

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 256 entries, 0 to 255
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   employee_name       256 non-null    str    
 1   department          256 non-null    str    
 2   years_experience    256 non-null    float64
 3   salary              256 non-null    str    
 4   satisfaction_score  247 non-null    float64
 5   hire_date           256 non-null    str    
dtypes: float64(2), str(4)
memory usage: 12.1 KB


In [3]:
print("Duplicate rows:", df.duplicated().sum())
print("\nMissing values per column:\n", df.isna().sum())
print("\nUnique department values (before cleaning):", sorted(df['department'].unique()))

Duplicate rows: 6

Missing values per column:
 employee_name         0
department            0
years_experience      0
salary                0
satisfaction_score    9
hire_date             0
dtype: int64

Unique department values (before cleaning): [' Engineering', 'ENGINEERING', 'Engineering', 'HR', 'Hr', 'Marketing', 'Sales', 'Sales ', 'engineering', 'hr', 'marketing', 'sales']


## 2. Cleaning Steps

1. Standardize `department` text (strip whitespace, consistent casing).
2. Drop exact duplicate rows.
3. Convert `salary` from a comma-formatted string to numeric.
4. Parse `hire_date` as a real datetime.
5. Fill missing `satisfaction_score` values with the column median.
6. Convert `department` to a `category` dtype.

In [4]:
df["department"] = df["department"].str.strip().str.title()
# "Hr" isn't how the acronym should read - fix it explicitly after title-casing
df["department"] = df["department"].replace({"Hr": "HR"})
print("Departments after standardizing:", sorted(df["department"].unique()))

Departments after standardizing: ['Engineering', 'HR', 'Marketing', 'Sales']


In [5]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)
print(f"Removed {before - after} duplicate rows ({before} -> {after})")

Removed 6 duplicate rows (256 -> 250)


In [6]:
df["salary"] = df["salary"].str.replace(",", "", regex=False).astype(float)
df["hire_date"] = pd.to_datetime(df["hire_date"])
df["department"] = df["department"].astype("category")
df.dtypes

employee_name                    str
department                  category
years_experience             float64
salary                       float64
satisfaction_score           float64
hire_date             datetime64[us]
dtype: object

In [7]:
median_satisfaction = df["satisfaction_score"].median()
missing_before = df["satisfaction_score"].isna().sum()
df["satisfaction_score"] = df["satisfaction_score"].fillna(median_satisfaction)
print(f"Filled {missing_before} missing satisfaction_score values with median ({median_satisfaction})")

Filled 8 missing satisfaction_score values with median (7.1)


## 3. Outlier Check (Salary, IQR Method)

In [8]:
Q1 = df["salary"].quantile(0.25)
Q3 = df["salary"].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

outliers = df[(df["salary"] < lower) | (df["salary"] > upper)]
print(f"Salary IQR bounds: [{lower:,.0f}, {upper:,.0f}]")
print(f"Outliers detected: {len(outliers)} of {len(df)} rows")

plt.figure(figsize=(6, 5))
sns.boxplot(y=df["salary"], color="#a6cee3")
plt.title("Salary Distribution (Box Plot)")
plt.ylabel("Salary ($)")
plt.tight_layout()
plt.savefig("../plots/salary_boxplot_W3D5.png", dpi=150)
plt.show()

Salary IQR bounds: [35,509, 129,736]
Outliers detected: 6 of 250 rows


These outliers are kept in the analysis rather than dropped — they
represent real high earners, not data-entry errors, so removing them
would hide a true feature of the salary distribution rather than clean
up a mistake.

## 4. Exploratory Charts

In [9]:
plt.figure(figsize=(7, 5))
dept_counts = df["department"].value_counts()
sns.barplot(x=dept_counts.index, y=dept_counts.values, hue=dept_counts.index,
            palette="Blues_d", legend=False)
plt.title("Employee Count by Department")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("../plots/department_counts_W3D5.png", dpi=150)
plt.show()

In [10]:
plt.figure(figsize=(7, 5))
sns.histplot(df["salary"], kde=True, color="#55a868")
plt.axvline(df["salary"].mean(), color="red", linestyle="--", label=f"Mean = {df['salary'].mean():,.0f}")
plt.axvline(df["salary"].median(), color="green", linestyle="-", label=f"Median = {df['salary'].median():,.0f}")
plt.title("Salary Distribution")
plt.xlabel("Salary ($)")
plt.legend()
plt.tight_layout()
plt.savefig("../plots/salary_distribution_W3D5.png", dpi=150)
plt.show()

In [11]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="years_experience", y="salary", hue="department", alpha=0.7)
plt.title("Salary vs. Years of Experience")
plt.xlabel("Years of Experience")
plt.ylabel("Salary ($)")
plt.tight_layout()
plt.savefig("../plots/salary_vs_experience_W3D5.png", dpi=150)
plt.show()

In [12]:
plt.figure(figsize=(6, 5))
corr = df[["years_experience", "salary", "satisfaction_score"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig("../plots/correlation_heatmap_W3D5.png", dpi=150)
plt.show()

In [13]:
plt.figure(figsize=(7, 5))
avg_by_dept = df.groupby("department", observed=True)["salary"].mean().sort_values(ascending=False)
sns.barplot(x=avg_by_dept.index, y=avg_by_dept.values, hue=avg_by_dept.index,
            palette="magma", legend=False)
plt.title("Average Salary by Department")
plt.ylabel("Average Salary ($)")
plt.tight_layout()
plt.savefig("../plots/avg_salary_by_department_W3D5.png", dpi=150)
plt.show()

## 5. Key Findings

In [14]:
dept_counts = df["department"].value_counts()
avg_by_dept = df.groupby("department", observed=True)["salary"].mean().sort_values(ascending=False)
corr_exp_salary = df["years_experience"].corr(df["salary"])
corr_sal_satisfaction = df["salary"].corr(df["satisfaction_score"])


def describe_strength(r):
    r_abs = abs(r)
    if r_abs >= 0.7:
        return "strongly"
    elif r_abs >= 0.4:
        return "moderately"
    elif r_abs >= 0.2:
        return "weakly"
    else:
        return "negligibly"


print("=== Key Findings ===\n")
print(f"1. {dept_counts.index[0]} is the largest department ({dept_counts.iloc[0]} employees), "
      f"{dept_counts.index[-1]} the smallest ({dept_counts.iloc[-1]}).")
print(f"2. Salary is right-skewed: mean (${df['salary'].mean():,.0f}) exceeds "
      f"median (${df['salary'].median():,.0f}), and the IQR method flagged "
      f"{len(outliers)} high-salary outliers.")
print(f"3. Salary is only {describe_strength(corr_exp_salary)} correlated with years "
      f"of experience (r = {corr_exp_salary:.2f}) - department and the built-in "
      f"randomness in this sample matter as much or more than tenure alone.")
print(f"4. {avg_by_dept.index[0]} pays the highest average salary "
      f"(${avg_by_dept.iloc[0]:,.0f}), {avg_by_dept.index[-1]} the lowest "
      f"(${avg_by_dept.iloc[-1]:,.0f}).")
print(f"5. Satisfaction score is only weakly related to salary "
      f"(r = {corr_sal_satisfaction:.2f}) - pay isn't the main driver of "
      f"reported satisfaction in this dataset.")

=== Key Findings ===

1. Engineering is the largest department (97 employees), HR the smallest (22).
2. Salary is right-skewed: mean ($84,092) exceeds median ($82,012), and the IQR method flagged 6 high-salary outliers.
3. Salary is only weakly correlated with years of experience (r = 0.23) - department and the built-in randomness in this sample matter as much or more than tenure alone.
4. Engineering pays the highest average salary ($98,397), HR the lowest ($65,750).
5. Satisfaction score is only weakly related to salary (r = 0.07) - pay isn't the main driver of reported satisfaction in this dataset.
